# Demonstração de Resultados Pro Forma Trimestral com PROC COMPUTAB

## Resumo Executivo

Este notebook constrói a demonstração de resultados pro forma trimestral de um banco regional diretamente a partir dos dados mensais do razão usando o PROC COMPUTAB, o procedimento de geração de tabelas e relatórios do SAS/ETS. Ele encaminha a receita de juros, a despesa de juros, a receita de tarifas e os custos operacionais de cada mês para a coluna do trimestre-calendário correto e, em seguida, usa blocos de programação de linha para calcular a receita líquida de juros, o resultado antes de impostos e o lucro líquido, além de um bloco de coluna para consolidar os quatro trimestres em um total anual.

## Fontes de Dados

O único conjunto de dados sintético `bank_ledger` simula um exercício fiscal de itens mensais da demonstração financeira de um banco comunitário de porte médio. Doze observações mensais são geradas inline com `call streaminit`/`rand`, de modo que o notebook seja totalmente autossuficiente.

| Variável | Tipo | Descrição |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Data de fechamento do mês da demonstração (jan–dez) |
| `loanint`  | Num | Juros e tarifas auferidos na carteira de crédito (milhares de USD) |
| `secint`   | Num | Juros auferidos na carteira de títulos de investimento (milhares de USD) |
| `depint`   | Num | Juros pagos sobre depósitos (milhares de USD) |
| `borrint`  | Num | Juros pagos sobre empréstimos / adiantamentos do FHLB (milhares de USD) |
| `feeinc`   | Num | Receita não decorrente de juros (tarifas e cobranças de serviços) (milhares de USD) |
| `salaries` | Num | Despesa com salários e benefícios de funcionários (milhares de USD) |
| `occupancy`| Num | Despesa com ocupação e equipamentos (milhares de USD) |
| `othropex` | Num | Outras despesas operacionais não decorrentes de juros (milhares de USD) |
| `provision`| Num | Provisão para perdas de crédito (milhares de USD) |
| `taxrate`  | Num | Alíquota efetiva de imposto aplicada ao resultado antes de impostos |

# Demonstração de Resultados Pro Forma Trimestral com PROC COMPUTAB

As equipes financeiras de bancos rotineiramente transformam um razão geral mensal em uma **demonstração de resultados trimestral** que apresenta a receita líquida de juros e o lucro líquido final. O `PROC COMPUTAB` (SAS/ETS) foi criado especificamente para isso: ele dispõe uma tabela programável cujas **colunas** são os períodos de reporte e cujas **linhas** são os itens da demonstração, e permite calcular subtotais com a lógica comum do DATA step em blocos de linha e de coluna.

Neste notebook, nós:

1. Geramos um exercício fiscal de dados mensais sintéticos do razão para um banco comunitário.
2. Encaminhamos cada mês para o seu trimestre-calendário e construímos uma demonstração de resultados trimestral.
3. Calculamos a receita líquida de juros, o resultado antes de impostos e o lucro líquido em um **bloco de linha**, e consolidamos os trimestres em um total anual em um **bloco de coluna**.
4. Reutilizamos a tabela `out=` para que a demonstração calculada possa alimentar relatórios subsequentes.

## Etapa 1 — Gerar dados mensais sintéticos do razão

Simulamos doze observações de fechamento de mês. A receita de juros de crédito e de títulos apresenta uma leve tendência de alta ao longo do ano, os custos de depósitos e empréstimos acompanham o ambiente de taxas, e a receita de tarifas, as despesas operacionais e a provisão para perdas de crédito carregam um ruído realista de mês a mês. O `call streaminit` fixa a semente para que os dados sejam reproduzíveis.

In [1]:
DADOS bank_ledger;
   CHAMAR streaminit(20240115);
   FORMATO stmtdate date9.;
   FAZER month = 1 ATÉ 12;
      /* Data de fechamento do mês para o exercício fiscal de 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Leve tendência de alta ao longo do ano + ruído mensal */
      drift = 1 + 0.012 * (month - 1);

      /* Receita de juros (milhares de USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Despesa de juros (milhares de USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Receita e despesa não decorrentes de juros (milhares de USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provisão para perdas de crédito, ocasionalmente elevada */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Alíquota efetiva de imposto */
      taxrate = 0.21;

      SAÍDA;
   FIM;
   MANTER stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=bank_ledger noobs RÓTULO;
   TÍTULO "Razão Mensal Sintético — Banco Comunitário, Exercício Fiscal de 2024";
   RÓTULO stmtdate="Data da Demonstração"
         loanint="Juros e Tarifas sobre Empréstimos"
         secint="Juros sobre Títulos"
         depint="Juros sobre Depósitos"
         borrint="Juros sobre Empréstimos Tomados"
         feeinc="Receita de Tarifas"
         salaries="Salários e Benefícios"
         occupancy="Ocupação e Equipamentos"
         othropex="Outras Despesas Operacionais"
         provision="Provisão para Perdas de Crédito"
         taxrate="Alíquota Efetiva de Imposto";
   FORMATO loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
EXECUTAR;

                          Razão Mensal Sintético — Banco Comunitário, Exercício Fiscal de 2024                          

  Data da Demonstração   Juros e Tarifas sobre Empréstimos   Juros sobre Títulos   Juros sobre Depósitos   Juros sobre Empréstimos Tomados  Receita de Tarifas    Salários e Benefícios    Ocupação e Equipamentos  Outras Despesas Operacionais    Provisão para Perdas de Crédito   Alíquota Efetiva de Imposto
             28JAN2024                            1,772.95                423.79                  531.47                            128.99              339.33                   699.38                     171.95                        202.43                             130.93                          0.21
             28FEB2024                            1,824.38                421.13                  564.85                            125.53              294.09                   767.29                     162.79                        307.61                          


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Etapa 2 — Construir a demonstração de resultados trimestral

O núcleo do relatório é a etapa `PROC COMPUTAB` abaixo.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** define quatro colunas de trimestre mais uma coluna de ano completo.
- O **bloco de entrada** sem rótulo define a variável automática **`_col_`** com `qtr(stmtdate)`, o que encaminha cada observação mensal para a coluna do trimestre correta. Como a entrada é transposta por padrão, cada variável do razão cai na linha que compartilha o seu nome.
- O **bloco de linha** `inc_stmt:` é executado uma vez por coluna e calcula as linhas derivadas — receita líquida de juros, despesa total não decorrente de juros, resultado antes de impostos e lucro líquido — exatamente como faria um contador.
- O **bloco de coluna** `total:` é executado uma vez por linha e soma os quatro trimestres na coluna `fy` (ano completo).

As instruções `rows ... / ...` adicionam títulos, recuos e linhas de régua (`ol` linha superior, `ul` linha inferior, `dul` linha inferior dupla) para que a saída seja lida como uma demonstração financeira de verdade.

In [2]:
TÍTULO "Demonstração de Resultados Pro Forma — Community Bancorp, Inc.";
title2 "Exercício Fiscal de 2024  (Valores em Milhares de USD)";

PROCEDIMENTO computab DADOS=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Quatro trimestres mais uma coluna consolidada de ano completo */
   columns qtr1 qtr2 qtr3 qtr4 fy / FORMATO=comma11.1;
   columns qtr1 / 'T1';
   columns qtr2 / 'T2';
   columns qtr3 / 'T3';
   columns qtr4 / 'T4';
   columns fy   / 'Ano' 'Completo' +3;

   /* Linhas da demonstração de resultados, de cima para baixo */
   rows loanint  / "Juros e Tarifas sobre Empréstimos";
   rows secint   / "Juros sobre Títulos"              ul;
   rows intinc   / "Receita Total de Juros";
   rows depint   / "Juros sobre Depósitos";
   rows borrint  / "Juros sobre Empréstimos Tomados"  ul;
   rows intexp   / "Despesa Total de Juros";
   rows nii      / "Receita Líquida de Juros"         ol skip;
   rows provision/ "Provisão para Perdas de Crédito"  ul;
   rows niiap    / "Rec. Líq. de Juros após Prov."    skip;
   rows feeinc   / "Receita Não Decorrente de Juros"  ul;
   rows salaries / "Salários e Benefícios";
   rows occupancy/ "Ocupação e Equipamentos";
   rows othropex / "Outras Despesas Operacionais"     ul;
   rows nonintexp/ "Despesa Total Não Dec. de Juros"  skip;
   rows pretax   / "Resultado Antes de Impostos"      ol;
   rows taxexp   / "Despesa de Imposto de Renda"      ul;
   rows netinc   / "Lucro Líquido"                    dul;

   /* Bloco de entrada: encaminha cada mês para o seu trimestre-calendário */
   _col_ = qtr(stmtdate);

   /* Bloco de linha: calcula os subtotais da demonstração em cada coluna */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Bloco de coluna: consolida os quatro trimestres na coluna de ano completo */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
EXECUTAR;

                             Demonstração de Resultados Pro Forma — Community Bancorp, Inc.                             
                                 Exercício Fiscal de 2024  (Valores em Milhares de USD)                                 


                             The COMPUTAB Procedure                             

                             T1           T2           T3           T4          Ano  
                                                                           Completo  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Juros e Tarifas sobre Empréstimos
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Juros sobre Títulos
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Receita Tota


NOTE: Option TITLE changed to Demonstração de Resultados Pro Forma — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Exercício Fiscal de 2024  (Valores em Milhares de USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Etapa 3 — Reutilizar o conjunto de dados de saída do COMPUTAB

A etapa `PROC COMPUTAB` acima gravou sua tabela calculada em `qtr_income` por meio de `out=`. Cada linha desse conjunto de dados é um item da demonstração e cada variável de coluna (`qtr1`–`qtr4`, `fy`) contém o valor calculado, de modo que pode alimentar relatórios subsequentes. Abaixo, imprimimos a coluna consolidada de ano completo para confirmar que os valores fecham.

In [3]:
TÍTULO "Conjunto de Dados de Saída do COMPUTAB — Coluna de Ano Completo";

PROCEDIMENTO IMPRIMIR DADOS=qtr_income RÓTULO noobs;
   VARIÁVEL _row_ fy;
   FORMATO fy comma12.1;
   RÓTULO _row_ = "Item da Demonstração" fy = "Ano Completo";
EXECUTAR;

TÍTULO;

                            Conjunto de Dados de Saída do COMPUTAB — Coluna de Ano Completo                             
                                 Exercício Fiscal de 2024  (Valores em Milhares de USD)                                 


  Item da Demonstração  Ano Completo
----------------------  ------------
loanint                     23,588.4
secint                       5,416.8
intinc                      29,005.2
depint                       6,831.2
borrint                      1,650.7
intexp                       8,482.0
nii                         20,523.2
provision                    1,568.9
niiap                       18,954.3
feeinc                       3,703.2
salaries                     8,763.1
occupancy                    1,985.6
othropex                     2,944.8
nonintexp                   13,693.5
pretax                       8,964.1
taxexp                       1,882.5
netinc                       7,081.6




NOTE: Option TITLE changed to Conjunto de Dados de Saída do COMPUTAB — Coluna de Ano Completo.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretando os resultados

A demonstração pro forma é lida de cima para baixo como o call report regulatório de um banco: a receita total de juros menos a despesa total de juros resulta na **receita líquida de juros** — aqui cerca de \$20,5 milhões no ano — o principal motor de resultados do banco. Subtraindo a **provisão para perdas de crédito**, somando a **receita de tarifas** e deduzindo a **despesa não decorrente de juros**, obtém-se o resultado antes de impostos de aproximadamente \$9,0 milhões, e aplicando a alíquota efetiva de imposto de 21% produz-se o **lucro líquido** próximo de \$7,1 milhões. O bloco de coluna `total:` soma os quatro trimestres na coluna de ano completo, de modo que os totais anuais se reconciliam com a soma dos trimestres por construção — confirmado no conjunto de dados `out=`, onde o `netinc` de ano completo de 7.081,6 é igual à soma dos quatro valores trimestrais.

Como tudo é calculado dentro dos blocos programáveis de linha e de coluna do `PROC COMPUTAB`, substituir por um razão mensal real não exige nenhuma alteração na lógica do relatório — apenas o conjunto de dados de entrada muda. O conjunto de dados `out=` disponibiliza então a demonstração calculada para painéis, análise de tendências ou automação de material para o conselho.